# 02 — LightGCN for the Recommender-System Challenge

This notebook implements a practical LightGCN model with BPR loss for implicit-feedback top-N recommendation.

It is safe to run at the same time as the EDA and SASRec notebooks: raw files are read-only, and outputs/checkpoints are written to a unique `outputs/lightgcn_<run_id>/` directory.

Recommended workflow:
1. Run validation training on the temporal split.
2. Tune the hyperparameters in the configuration cell.
3. Set `RUN_FINAL_TRAINING = True` to train on all deduplicated interactions and generate a Kaggle submission.

In [34]:
from pathlib import Path
from datetime import datetime, timezone
import uuid
import math
import random
import json
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

DATA_DIR = Path('data/')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
OUTPUT_DIR = Path('outputs') / f'lightgcn_{RUN_ID}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DATA_DIR =', DATA_DIR.resolve())
print('OUTPUT_DIR =', OUTPUT_DIR.resolve())
print('DEVICE =', DEVICE)

DATA_DIR = /home/joris/Master/Semester 2/Recommender Systems/Final_Assignment_RS/data
OUTPUT_DIR = /home/joris/Master/Semester 2/Recommender Systems/Final_Assignment_RS/outputs/lightgcn_20260609_183026_35d94d
DEVICE = cuda


In [35]:
# Core configuration.
# FAST_DEV_RUN keeps the notebook quick for smoke testing. Turn it off for leaderboard training.
FAST_DEV_RUN = False
RUN_FINAL_TRAINING = True

CONFIG = {
    'embedding_dim': 256,
    'n_layers': 4,
    'lr': 1e-3,
    'weight_decay': 0.0,
    'bpr_reg': 1e-5,
    'batch_size': 4096,
    'epochs': 4 if FAST_DEV_RUN else 100,
    'eval_every': 1,
    'patience': 20,
    'num_workers': 0,
    'popularity_blend_alpha': 0.03,  # add small normalized log-pop score at inference
}
CONFIG

{'embedding_dim': 256,
 'n_layers': 4,
 'lr': 0.001,
 'weight_decay': 0.0,
 'bpr_reg': 1e-05,
 'batch_size': 4096,
 'epochs': 100,
 'eval_every': 1,
 'patience': 20,
 'num_workers': 0,
 'popularity_blend_alpha': 0.03}

In [36]:
def load_data(data_dir=DATA_DIR):
    train = pd.read_csv(data_dir / 'train.csv')
    test = pd.read_csv(data_dir / 'test.csv')
    sample = pd.read_csv(data_dir / 'sample_submission.csv')
    train['datetime'] = pd.to_datetime(train['timestamp'], unit='ms', utc=True)
    test['datetime'] = pd.to_datetime(test['timestamp'], unit='ms', utc=True)
    return train, test, sample

train_raw, test_raw, sample = load_data()

# Drop duplicate user-item positives. In this data, repeated pairs have the same timestamp, so they are accidental duplicates.
train_all = train_raw.sort_values('timestamp').drop_duplicates(['user_id', 'item_id'], keep='first').copy()
cutoff = test_raw['timestamp'].min()
train_fit = train_all[train_all['timestamp'] < cutoff].copy()
valid = train_all[train_all['timestamp'] >= cutoff].copy()

print('train_all:', train_all.shape)
print('train_fit:', train_fit.shape)
print('valid:', valid.shape)
print('sample users:', sample['user_id'].nunique())

train_all: (158471, 4)
train_fit: (143313, 4)
valid: (15158, 4)
sample users: 2255


In [37]:
class InteractionData:
    """Holds integer mappings and sparse structures for a LightGCN training split."""
    def __init__(self, interactions: pd.DataFrame):
        self.raw_users = np.sort(interactions['user_id'].unique())
        self.raw_items = np.sort(interactions['item_id'].unique())
        self.user2idx = {u: i for i, u in enumerate(self.raw_users)}
        self.idx2user = {i: u for u, i in self.user2idx.items()}
        self.item2idx = {it: i for i, it in enumerate(self.raw_items)}
        self.idx2item = {i: it for it, i in self.item2idx.items()}
        self.n_users = len(self.raw_users)
        self.n_items = len(self.raw_items)

        df = interactions[['user_id', 'item_id']].copy()
        df['u'] = df['user_id'].map(self.user2idx)
        df['i'] = df['item_id'].map(self.item2idx)
        self.df = df.dropna(subset=['u', 'i']).astype({'u': 'int64', 'i': 'int64'})
        self.user_indices = self.df['u'].to_numpy(np.int64)
        self.item_indices = self.df['i'].to_numpy(np.int64)
        self.seen = self.df.groupby('u')['i'].apply(set).to_dict()

    def map_validation_targets(self, valid_df: pd.DataFrame):
        tmp = valid_df[['user_id', 'item_id']].copy()
        tmp['u'] = tmp['user_id'].map(self.user2idx)
        tmp['i'] = tmp['item_id'].map(self.item2idx)
        tmp = tmp.dropna(subset=['u', 'i']).astype({'u': 'int64', 'i': 'int64'})
        return tmp.groupby('u')['i'].apply(set).to_dict()

class BPRDataset(Dataset):
    def __init__(self, data: InteractionData, seed=42):
        self.data = data
        self.users = data.user_indices
        self.pos_items = data.item_indices
        self.n_items = data.n_items
        self.seen = data.seen
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.users)

    def sample_negative(self, u):
        # Rejection sampling is fine at this sparsity.
        seen_u = self.seen.get(int(u), set())
        while True:
            j = int(self.rng.integers(0, self.n_items))
            if j not in seen_u:
                return j

    def __getitem__(self, idx):
        u = int(self.users[idx])
        i = int(self.pos_items[idx])
        j = self.sample_negative(u)
        return torch.tensor(u, dtype=torch.long), torch.tensor(i, dtype=torch.long), torch.tensor(j, dtype=torch.long)

In [38]:
def build_normalized_adj(data: InteractionData, device=DEVICE):
    """Build D^{-1/2} A D^{-1/2} for the bipartite user-item graph."""
    n_nodes = data.n_users + data.n_items
    users = data.user_indices
    items = data.item_indices + data.n_users

    row = np.concatenate([users, items])
    col = np.concatenate([items, users])
    vals = np.ones(len(row), dtype=np.float32)

    adj = sp.coo_matrix((vals, (row, col)), shape=(n_nodes, n_nodes))
    deg = np.asarray(adj.sum(axis=1)).ravel()
    deg_inv_sqrt = np.zeros_like(deg, dtype=np.float32)
    mask = deg > 0
    deg_inv_sqrt[mask] = np.power(deg[mask], -0.5)
    norm_vals = deg_inv_sqrt[row] * vals * deg_inv_sqrt[col]

    indices = torch.tensor(np.vstack([row, col]), dtype=torch.long)
    values = torch.tensor(norm_vals, dtype=torch.float32)
    sparse = torch.sparse_coo_tensor(indices, values, size=(n_nodes, n_nodes)).coalesce()
    return sparse.to(device)

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64, n_layers=3):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.embedding_dim = embedding_dim
        self.n_layers = n_layers
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def propagate(self, norm_adj):
        all_emb = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        embs = [all_emb]
        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)
        final_emb = torch.stack(embs, dim=0).mean(dim=0)
        users, items = torch.split(final_emb, [self.n_users, self.n_items], dim=0)
        return users, items

    def bpr_scores(self, users, pos_items, neg_items, norm_adj):
        user_emb, item_emb = self.propagate(norm_adj)
        u = user_emb[users]
        pos = item_emb[pos_items]
        neg = item_emb[neg_items]
        pos_scores = (u * pos).sum(dim=1)
        neg_scores = (u * neg).sum(dim=1)
        return pos_scores, neg_scores, u, pos, neg

    @torch.no_grad()
    def full_sort_scores(self, user_ids, norm_adj, batch_items=None):
        user_emb, item_emb = self.propagate(norm_adj)
        return user_emb[user_ids] @ item_emb.T

def bpr_loss(pos_scores, neg_scores, u_emb, pos_emb, neg_emb, reg=1e-5):
    ranking_loss = -F.logsigmoid(pos_scores - neg_scores).mean()
    reg_loss = (u_emb.norm(2).pow(2) + pos_emb.norm(2).pow(2) + neg_emb.norm(2).pow(2)) / (2.0 * len(pos_scores))
    return ranking_loss + reg * reg_loss

In [39]:
@torch.no_grad()
def recall_ndcg_at_k(model, data: InteractionData, norm_adj, targets, k=10, batch_size=512, popularity_scores=None, alpha=0.0):
    model.eval()
    users = sorted(targets.keys())
    recalls, ndcgs = [], []

    if popularity_scores is not None:
        pop = torch.tensor(popularity_scores, dtype=torch.float32, device=DEVICE)
    else:
        pop = None

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        u_tensor = torch.tensor(batch_users, dtype=torch.long, device=DEVICE)
        scores = model.full_sort_scores(u_tensor, norm_adj)
        if pop is not None and alpha > 0:
            scores = scores + alpha * pop.unsqueeze(0)

        # Mask items already seen in the training graph.
        for row, u in enumerate(batch_users):
            seen_u = data.seen.get(u, set())
            if seen_u:
                scores[row, torch.tensor(list(seen_u), device=DEVICE)] = -float('inf')

        topk = torch.topk(scores, k=k, dim=1).indices.cpu().numpy()
        for row, u in enumerate(batch_users):
            true_items = targets[u]
            pred = list(topk[row])
            hits = [item for item in pred if item in true_items]
            recalls.append(len(hits) / min(k, len(true_items)))
            if hits:
                dcg = sum(1.0 / math.log2(pred.index(item) + 2) for item in hits)
                ideal = sum(1.0 / math.log2(r + 2) for r in range(min(k, len(true_items))))
                ndcgs.append(dcg / ideal)
            else:
                ndcgs.append(0.0)

    return {'Recall@10': float(np.mean(recalls)) if recalls else 0.0,
            'NDCG@10': float(np.mean(ndcgs)) if ndcgs else 0.0,
            'num_eval_users': len(users)}

def make_popularity_scores(data: InteractionData):
    counts = np.bincount(data.item_indices, minlength=data.n_items).astype(np.float32)
    scores = np.log1p(counts)
    if scores.max() > 0:
        scores = scores / scores.max()
    return scores

In [40]:
def train_lightgcn(train_df, valid_df=None, config=CONFIG):
    data = InteractionData(train_df)
    norm_adj = build_normalized_adj(data, DEVICE)
    dataset = BPRDataset(data, seed=SEED)
    loader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True,
                        num_workers=config['num_workers'], pin_memory=(DEVICE.type == 'cuda'))
    model = LightGCN(data.n_users, data.n_items, config['embedding_dim'], config['n_layers']).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    pop_scores = make_popularity_scores(data)

    targets = data.map_validation_targets(valid_df) if valid_df is not None else {}
    best_score = -1.0
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, config['epochs'] + 1):
        model.train()
        t0 = time.time()
        total_loss = 0.0
        for users, pos, neg in loader:
            users = users.to(DEVICE, non_blocking=True)
            pos = pos.to(DEVICE, non_blocking=True)
            neg = neg.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pos_scores, neg_scores, u_emb, pos_emb, neg_emb = model.bpr_scores(users, pos, neg, norm_adj)
            loss = bpr_loss(pos_scores, neg_scores, u_emb, pos_emb, neg_emb, reg=config['bpr_reg'])
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(users)

        row = {'epoch': epoch, 'loss': total_loss / len(dataset), 'seconds': time.time() - t0}

        if targets and (epoch % config['eval_every'] == 0):
            metrics = recall_ndcg_at_k(model, data, norm_adj, targets, k=10,
                                       popularity_scores=pop_scores,
                                       alpha=config['popularity_blend_alpha'])
            row.update(metrics)
            print(f"epoch {epoch:03d} loss={row['loss']:.5f} recall10={metrics['Recall@10']:.5f} ndcg10={metrics['NDCG@10']:.5f} time={row['seconds']:.1f}s")
            score = metrics['Recall@10']
            if score > best_score:
                best_score = score
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= config['patience']:
                    print('early stopping')
                    break
        else:
            print(f"epoch {epoch:03d} loss={row['loss']:.5f} time={row['seconds']:.1f}s")
        history.append(row)

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, data, norm_adj, pop_scores, pd.DataFrame(history)

In [41]:
# Validation run.
model, data, norm_adj, pop_scores, history = train_lightgcn(train_fit, valid, CONFIG)
history.to_csv(OUTPUT_DIR / 'lightgcn_validation_history.csv', index=False)
display(history.tail())

valid_targets = data.map_validation_targets(valid)
final_valid_metrics = recall_ndcg_at_k(model, data, norm_adj, valid_targets, k=10,
                                       popularity_scores=pop_scores,
                                       alpha=CONFIG['popularity_blend_alpha'])
print('Final validation metrics:', final_valid_metrics)
with open(OUTPUT_DIR / 'lightgcn_validation_metrics.json', 'w') as f:
    json.dump(final_valid_metrics, f, indent=2)

epoch 001 loss=0.67148 recall10=0.00501 ndcg10=0.00297 time=1.6s
epoch 002 loss=0.65636 recall10=0.00945 ndcg10=0.00608 time=1.7s
epoch 003 loss=0.61577 recall10=0.01232 ndcg10=0.00731 time=2.2s
epoch 004 loss=0.54068 recall10=0.01257 ndcg10=0.00753 time=2.0s
epoch 005 loss=0.45113 recall10=0.01364 ndcg10=0.00803 time=2.1s
epoch 006 loss=0.37692 recall10=0.01478 ndcg10=0.00832 time=2.0s
epoch 007 loss=0.33206 recall10=0.01439 ndcg10=0.00824 time=2.4s
epoch 008 loss=0.30371 recall10=0.01401 ndcg10=0.00816 time=2.2s
epoch 009 loss=0.28465 recall10=0.01406 ndcg10=0.00818 time=2.1s
epoch 010 loss=0.26885 recall10=0.01361 ndcg10=0.00801 time=2.2s
epoch 011 loss=0.25306 recall10=0.01336 ndcg10=0.00798 time=2.0s
epoch 012 loss=0.23944 recall10=0.01347 ndcg10=0.00806 time=2.4s
epoch 013 loss=0.22596 recall10=0.01359 ndcg10=0.00814 time=2.0s
epoch 014 loss=0.21407 recall10=0.01349 ndcg10=0.00815 time=2.1s
epoch 015 loss=0.20183 recall10=0.01385 ndcg10=0.00831 time=2.2s
epoch 016 loss=0.18937 re

,epoch,loss,seconds,Recall@10,NDCG@10,num_eval_users
20,21,0.140801,1.991660,0.013263,0.007992,7437
21,22,0.131380,2.146113,0.013781,0.008162,7437
22,23,0.124932,1.959518,0.013955,0.008276,7437
23,24,0.117479,2.066580,0.014121,0.008333,7437
24,25,0.111349,2.050304,0.014053,0.008354,7437


Final validation metrics: {'Recall@10': 0.014775329700702836, 'NDCG@10': 0.008324516016378809, 'num_eval_users': 7437}


In [42]:
@torch.no_grad()
def recommend_for_raw_users(model, data: InteractionData, norm_adj, raw_users, k=10, popularity_scores=None, alpha=0.0):
    model.eval()
    recs = {}
    pop_rank = list(np.argsort(-(popularity_scores if popularity_scores is not None else np.ones(data.n_items))))
    pop_tensor = torch.tensor(popularity_scores, dtype=torch.float32, device=DEVICE) if popularity_scores is not None else None

    mapped = [(u, data.user2idx[u]) for u in raw_users if u in data.user2idx]
    missing = [u for u in raw_users if u not in data.user2idx]

    for start in range(0, len(mapped), 512):
        batch = mapped[start:start+512]
        raw_batch = [x[0] for x in batch]
        idx_batch = [x[1] for x in batch]
        u_tensor = torch.tensor(idx_batch, dtype=torch.long, device=DEVICE)
        scores = model.full_sort_scores(u_tensor, norm_adj)
        if pop_tensor is not None and alpha > 0:
            scores = scores + alpha * pop_tensor.unsqueeze(0)
        for row, u_idx in enumerate(idx_batch):
            seen_u = data.seen.get(u_idx, set())
            if seen_u:
                scores[row, torch.tensor(list(seen_u), device=DEVICE)] = -float('inf')
        topk = torch.topk(scores, k=k, dim=1).indices.cpu().numpy()
        for raw_u, item_idxs in zip(raw_batch, topk):
            recs[raw_u] = [int(data.idx2item[int(i)]) for i in item_idxs]

    # Fallback for users absent from the training graph.
    raw_pop = [int(data.idx2item[int(i)]) for i in pop_rank]
    for raw_u in missing:
        recs[raw_u] = raw_pop[:k]
    return recs

def write_submission(recs, sample_df, path):
    out = sample_df[['ID', 'user_id']].copy()
    out['item_id'] = out['user_id'].map(lambda u: ','.join(map(str, recs[int(u)][:10])))
    assert out['item_id'].str.split(',').map(len).eq(10).all()
    out.to_csv(path, index=False)
    return out

In [43]:
# Final training and submission generation.
# This trains on all deduplicated train interactions, then predicts for sample_submission users.
if RUN_FINAL_TRAINING:
    final_model, final_data, final_adj, final_pop_scores, final_history = train_lightgcn(train_all, valid_df=None, config=CONFIG)
    final_history.to_csv(OUTPUT_DIR / 'lightgcn_final_training_history.csv', index=False)
    torch.save(final_model.state_dict(), OUTPUT_DIR / 'lightgcn_final_model.pt')

    final_recs = recommend_for_raw_users(final_model, final_data, final_adj,
                                         raw_users=sample['user_id'].astype(int).tolist(),
                                         k=10,
                                         popularity_scores=final_pop_scores,
                                         alpha=CONFIG['popularity_blend_alpha'])
    submission = write_submission(final_recs, sample, OUTPUT_DIR / 'submission_lightgcn.csv')
    display(submission.head())
    print('Saved:', OUTPUT_DIR / 'submission_lightgcn.csv')
else:
    print('Set RUN_FINAL_TRAINING=True to train on all interactions and create the final LightGCN submission.')

epoch 001 loss=0.67411 time=2.5s
epoch 002 loss=0.65939 time=2.7s
epoch 003 loss=0.61623 time=2.4s
epoch 004 loss=0.53361 time=2.6s
epoch 005 loss=0.44300 time=2.3s
epoch 006 loss=0.37606 time=2.5s
epoch 007 loss=0.33879 time=2.4s
epoch 008 loss=0.31434 time=2.5s
epoch 009 loss=0.29697 time=2.3s
epoch 010 loss=0.28109 time=2.5s
epoch 011 loss=0.26553 time=2.3s
epoch 012 loss=0.25285 time=2.6s
epoch 013 loss=0.23963 time=2.4s
epoch 014 loss=0.22691 time=2.5s
epoch 015 loss=0.21355 time=2.4s
epoch 016 loss=0.20426 time=2.4s
epoch 017 loss=0.19136 time=2.6s
epoch 018 loss=0.18117 time=2.4s
epoch 019 loss=0.17024 time=2.6s
epoch 020 loss=0.16055 time=2.2s
epoch 021 loss=0.15210 time=2.6s
epoch 022 loss=0.14252 time=2.5s
epoch 023 loss=0.13491 time=2.6s
epoch 024 loss=0.12809 time=2.3s
epoch 025 loss=0.12016 time=2.5s
epoch 026 loss=0.11361 time=2.4s
epoch 027 loss=0.10857 time=2.6s
epoch 028 loss=0.10145 time=2.3s
epoch 029 loss=0.09669 time=2.6s
epoch 030 loss=0.09052 time=2.2s
epoch 031 

,ID,user_id,item_id
0,12,12,"2140,10916,9580,4798,8421,4277,10608,4101,331,..."
1,14,14,"7637,9668,5149,2020,589,8981,9564,4698,1863,1487"
2,17,17,"7480,11461,5241,5352,8149,8093,5796,1328,669,5155"
3,21,21,"8421,10326,2140,10916,4798,331,6971,11384,4277..."
4,44,44,"1290,10607,789,5135,9962,11775,11024,11298,403..."


Saved: outputs/lightgcn_20260609_183026_35d94d/submission_lightgcn.csv


## Notes for improving LightGCN

- Tune `embedding_dim` in `{64, 128, 256}` and `n_layers` in `{2, 3, 4}`.
- Tune `popularity_blend_alpha`; LightGCN often benefits from a small popularity blend on sparse data.
- Add a time-decayed weighted graph if validation suggests recent interactions dominate.
- Use LightGCN scores as features in a later ranker or ensemble them with SASRec ranks.